# Authenticating with AzureCliCredential

## Overview

This notebook authenticates using `AzureCliCredential`, which signs you in with the identity from your Azure CLI session (the account you logged in with via `az login`). This is the recommended approach for local development because it reuses the sign-in you already have on your machine. Later notebooks use `DefaultAzureCredential`, a more general-purpose credential that automatically tries several authentication methods in sequence and is well suited to code that runs both locally and in Azure. This notebook will help you troubleshoot common authentication issues and ensure your setup is correct.

## Understanding AzureCliCredential

`AzureCliCredential` authenticates by requesting a token from the Azure CLI on your behalf. It relies entirely on the account you signed in with using `az login`, so make sure you have logged in to the correct tenant before running the cells below.

## Prerequisites

Ensure you have the following installed:
- Azure CLI
- Azure Developer CLI (optional)
- Python Virtual Environment or Conda (use `uv venv` or `conda create`)
- Required role assignments (Azure AI Developer)
- Jupyter Notebook environment - kernel configured to use Python 3.8 or later


# First, we'll authenticate using Azure CLI
This is the recommended approach for local development.

When you run the code below, you will be redirected to:
- Either the Azure portal in your browser to complete the login 
- Or use Windows login if you're already signed in to your machine

The code will:
1. Load environment variables from .env file, including the TENANT_ID
2. Use Azure CLI to log in to your specific tenant  
3. Test authentication by attempting to get a token


In [ ]:
# Import required packages
from IPython.display import display
from IPython.display import HTML
import getpass
from dotenv import load_dotenv
import os
from pathlib import Path  # For cross-platform path handling

# Get the path to the .env file which is in the parent directory
notebook_path = Path().absolute()  # Get absolute path of current notebook
parent_dir = notebook_path.parent  # Get parent directory
load_dotenv(parent_dir / '.env')  # Load environment variables from .env file

# Get tenant ID from environment variable
tenant_id = os.getenv("TENANT_ID")

# Azure login with specific tenant
!az login --tenant {tenant_id}

# Optionally pin a subscription. In the New Foundry there is no connection string to parse,
# so set AZURE_SUBSCRIPTION_ID in your .env if you want a specific subscription; otherwise
# the Azure CLI uses your default subscription from `az login`.
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")

if subscription_id:
    !az account set --subscription {subscription_id}
    print(f"✓ Successfully set subscription: {subscription_id}")
else:
    print("ℹ️ No AZURE_SUBSCRIPTION_ID set — using the default subscription from 'az login'.")


# Next, we'll test the authentication by attempting to get a token using AzureCliCredential

`AzureCliCredential` authenticates using the identity from your Azure CLI session — the account you signed in with using `az login`. It does not cycle through multiple credential types; it simply requests a token from the Azure CLI on your behalf. (The credential that tries several methods in sequence is `DefaultAzureCredential`, which the later notebooks use.)

The code below will:
1. Create an AzureCliCredential instance
2. Try to get a token for Azure Cognitive Services
3. Print a success message if the token is acquired

>Note: You may see some warning or debug log messages while the token is acquired - 
>this is normal and can be ignored as long as you see "Successfully acquired token!" at the end


In [ ]:
# Then use AzureCliCredential in your code
from azure.identity import AzureCliCredential
import logging

# Enable detailed logging
logging.basicConfig(level=logging.DEBUG)

try:
    credential = AzureCliCredential()
    # Test token acquisition
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    print("Successfully acquired token!")
except Exception as e:
    print(f"Authentication failed: {str(e)}")

## Now that you have successfully authenticated, you can proceed to [2-environment_setup.ipynb](2-environment_setup.ipynb), or try the additional authentication methods or troubleshoot below.


## Common Issues and Solutions

1. **Token Acquisition Failed**
   - Verify Azure CLI login: `az login --tenant <tenant-id>`
   - Check default subscription: `az account show`
   - Ensure correct tenant: `az account set --subscription <subscription-id>`

2. **Missing Role Assignments**
   - Add Azure AI Developer role:
   ```bash
   az role assignment create --assignee "user@domain.com" \
       --role "Azure AI Developer" \
       --scope "/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.CognitiveServices/accounts/<resource-name>"
   ```

3. **Environment Variable Issues**
   - Verify environment variables are set correctly
   - Check for typos in variable names
   - Ensure no extra spaces in values

## Best Practices

1. Always use environment variables for service principal credentials
2. Implement proper error handling and logging
3. Use managed identities when deploying to Azure services
4. Regularly rotate service principal secrets
5. Follow the principle of least privilege when assigning roles

## Additional Resources

- [Azure Identity Documentation](https://docs.microsoft.com/python/api/overview/azure/identity-readme)
- [DefaultAzureCredential Authentication Flow](https://docs.microsoft.com/azure/developer/python/azure-sdk-authenticate)
- [Azure RBAC Documentation](https://docs.microsoft.com/azure/role-based-access-control/overview)